# Data Audit and DeBERTa Setup

This notebook validates the BIPIA EmailQA training and test files, checks
context separation and duplicates, configures the DeBERTa tokenizer, and
verifies that a forward pass works. It is a preparation notebook; the final
paper-inspired evaluation is in `02_category_disjoint_cv_comparison.ipynb`.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/CSBP711_Assignment_2_Paper_Critique"
    )
else:
    PROJECT_DIR = Path.cwd().resolve().parent
    if not (PROJECT_DIR / "data").exists():
        PROJECT_DIR = Path.cwd().resolve()

DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project directory:", PROJECT_DIR)
print("Training file exists:", TRAIN_PATH.exists())
print("Testing file exists:", TEST_PATH.exists())


In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())

display(train_df.head(3))
display(test_df.head(3))

In [ ]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
LABEL_NAME_COLUMN = "label_name"


def inspect_dataset(df, dataset_name):
    print(f"\n--- {dataset_name} ---")

    print("\nLabel distribution:")
    print(df[LABEL_NAME_COLUMN].value_counts())

    pairs_per_context = df.groupby("context_id")[LABEL_COLUMN].agg(list)

    valid_pairs = pairs_per_context.apply(
        lambda labels: len(labels) == 2 and sorted(labels) == [0, 1]
    )

    print("\nNumber of unique contexts:", df["context_id"].nunique())
    print("Contexts with one benign and one attack sample:", valid_pairs.sum())
    print("Invalid context pairs:", (~valid_pairs).sum())

    word_counts = df[TEXT_COLUMN].astype(str).str.split().str.len()

    print("\nText length in words:")
    print(word_counts.describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

    print("\nLongest text preview:")
    longest_index = word_counts.idxmax()
    print(df.loc[longest_index, TEXT_COLUMN][:500])


inspect_dataset(train_df, "Training data")
inspect_dataset(test_df, "Testing data")

In [ ]:
train_contexts = set(train_df["context_id"])
test_contexts = set(test_df["context_id"])

train_texts = set(train_df[TEXT_COLUMN].astype(str))
test_texts = set(test_df[TEXT_COLUMN].astype(str))

print("Context ID overlap:", len(train_contexts.intersection(test_contexts)))
print("Exact text overlap:", len(train_texts.intersection(test_texts)))

print("\nDuplicate texts within training data:")
print(train_df[TEXT_COLUMN].duplicated().sum())

print("\nDuplicate texts within testing data:")
print(test_df[TEXT_COLUMN].duplicated().sum())

print("\nEmpty training texts:")
print((train_df[TEXT_COLUMN].astype(str).str.strip() == "").sum())

print("\nEmpty testing texts:")
print((test_df[TEXT_COLUMN].astype(str).str.strip() == "").sum())

train_categories = set(train_df["attack_category"].dropna())
test_categories = set(test_df["attack_category"].dropna())

print("\nAttack categories shared between train and test:")
print(sorted(train_categories.intersection(test_categories)))

print("\nModel input policy:")
print("Input column:", TEXT_COLUMN)
print("Target column:", LABEL_COLUMN)
print("Excluded metadata columns:")
print(["sample_id", "context_id", "split", "label_name", "attack_category", "attack_number"])

In [ ]:
def show_duplicate_texts(df, dataset_name):
    duplicate_rows = df[
        df.duplicated(subset=[TEXT_COLUMN], keep=False)
    ].copy()

    print(f"\n--- {dataset_name}: Duplicate text records ---")
    print("Number of rows involved:", len(duplicate_rows))
    print("Number of unique duplicated texts:", duplicate_rows[TEXT_COLUMN].nunique())

    display(
        duplicate_rows[
            [
                "sample_id",
                "context_id",
                TEXT_COLUMN,
                LABEL_COLUMN,
                LABEL_NAME_COLUMN,
                "attack_category",
                "attack_number",
            ]
        ].sort_values(TEXT_COLUMN)
    )

    label_counts = duplicate_rows.groupby(TEXT_COLUMN)[LABEL_COLUMN].nunique()

    print("\nDuplicated texts with conflicting labels:")
    print((label_counts > 1).sum())


show_duplicate_texts(train_df, "Training data")
show_duplicate_texts(test_df, "Testing data")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available.")

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

print("Tokenizer loaded successfully.")
print("Tokenizer maximum length:", tokenizer.model_max_length)

In [ ]:
!pip -q install -U "transformers" "datasets" "accelerate" "evaluate" "sentencepiece==0.2.1" "protobuf==7.36.1"

In [ ]:
from transformers import AutoConfig
import pandas as pd

model_config = AutoConfig.from_pretrained(MODEL_NAME)

MAX_LENGTH = model_config.max_position_embeddings

print("Model maximum position embeddings:", MAX_LENGTH)


def inspect_token_lengths(df, dataset_name):
    token_lengths = [
        len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
        for text in df[TEXT_COLUMN].astype(str)
    ]

    lengths = pd.Series(token_lengths)

    print(f"\n--- {dataset_name} token lengths ---")
    print(lengths.describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

    print(f"\nSequences longer than {MAX_LENGTH} tokens:")
    print((lengths > MAX_LENGTH).sum())

    longest_index = lengths.idxmax()

    print("\nLongest sequence details:")
    print("Sample ID:", df.loc[longest_index, "sample_id"])
    print("Context ID:", df.loc[longest_index, "context_id"])
    print("Label:", df.loc[longest_index, LABEL_NAME_COLUMN])
    print("Token length:", lengths.loc[longest_index])


inspect_token_lengths(train_df, "Training data")
inspect_token_lengths(test_df, "Testing data")

In [ ]:
MAX_LENGTH = 512

print("CLS token:", tokenizer.cls_token, tokenizer.cls_token_id)
print("SEP token:", tokenizer.sep_token, tokenizer.sep_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)


def head_tail_tokenize(text, tokenizer, max_length=MAX_LENGTH):
    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False,
    )

    max_content_length = max_length - 2

    if len(token_ids) > max_content_length:
        first_part_length = max_content_length // 2
        last_part_length = max_content_length - first_part_length

        token_ids = (
            token_ids[:first_part_length]
            + token_ids[-last_part_length:]
        )

    input_ids = (
        [tokenizer.cls_token_id]
        + token_ids
        + [tokenizer.sep_token_id]
    )

    attention_mask = [1] * len(input_ids)

    padding_length = max_length - len(input_ids)

    if padding_length > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
        attention_mask = attention_mask + [0] * padding_length

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }


for _, row in long_training_samples.iterrows():
    sample_text = train_df.loc[
        train_df["sample_id"] == row["sample_id"],
        TEXT_COLUMN,
    ].iloc[0]

    encoded_sample = head_tail_tokenize(sample_text, tokenizer)

    print(f"\nSample ID: {row['sample_id']}")
    print("Final encoded length:", len(encoded_sample["input_ids"]))
    print("Attention mask sum:", sum(encoded_sample["attention_mask"]))

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

all_train_context_ids = sorted(train_df["context_id"].unique())

rng = np.random.default_rng(SEED)

validation_context_ids = set(
    rng.choice(
        all_train_context_ids,
        size=10,
        replace=False,
    )
)

fit_train_df = train_df[
    ~train_df["context_id"].isin(validation_context_ids)
].copy()

validation_df = train_df[
    train_df["context_id"].isin(validation_context_ids)
].copy()

print("Fit training shape:", fit_train_df.shape)
print("Validation shape:", validation_df.shape)
print("Final test shape:", test_df.shape)

print("\nFit training label distribution:")
print(fit_train_df[LABEL_NAME_COLUMN].value_counts())

print("\nValidation label distribution:")
print(validation_df[LABEL_NAME_COLUMN].value_counts())

print("\nContext overlap between fit training and validation:")
print(
    len(
        set(fit_train_df["context_id"]).intersection(
            set(validation_df["context_id"])
        )
    )
)

print("\nContext overlap between validation and final test:")
print(
    len(
        set(validation_df["context_id"]).intersection(
            set(test_df["context_id"])
        )
    )
)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class EmailInjectionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_column, label_column):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_column = text_column
        self.label_column = label_column

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        encoded = head_tail_tokenize(
            row[self.text_column],
            self.tokenizer,
            max_length=MAX_LENGTH,
        )

        return {
            "input_ids": torch.tensor(
                encoded["input_ids"],
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                encoded["attention_mask"],
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                row[self.label_column],
                dtype=torch.long,
            ),
        }


BATCH_SIZE = 4

fit_train_dataset = EmailInjectionDataset(
    fit_train_df,
    tokenizer,
    TEXT_COLUMN,
    LABEL_COLUMN,
)

validation_dataset = EmailInjectionDataset(
    validation_df,
    tokenizer,
    TEXT_COLUMN,
    LABEL_COLUMN,
)

test_dataset = EmailInjectionDataset(
    test_df,
    tokenizer,
    TEXT_COLUMN,
    LABEL_COLUMN,
)

data_loader_generator = torch.Generator()
data_loader_generator.manual_seed(SEED)

fit_train_loader = DataLoader(
    fit_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=data_loader_generator,
    num_workers=0,
    pin_memory=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

first_batch = next(iter(fit_train_loader))

print("Training batches:", len(fit_train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

print("\nInput IDs shape:", first_batch["input_ids"].shape)
print("Attention mask shape:", first_batch["attention_mask"].shape)
print("Labels shape:", first_batch["labels"].shape)
print("Labels in first batch:", first_batch["labels"].tolist())

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

model.to(device)

print("Device:", device)
print("Model loaded successfully.")

smoke_batch = {
    key: value.to(device)
    for key, value in first_batch.items()
}

model.eval()

with torch.no_grad():
    smoke_output = model(**smoke_batch)

print("Logits shape:", smoke_output.logits.shape)
print("Initial loss:", round(smoke_output.loss.item(), 4))